# Guide 8: Fine-tuning: teaching a base model new behavior

*Lifecycle* · [Read the guide](https://drvgautam.github.io/primer-on-aiengineering/finetuning/) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/08-finetuning.ipynb)

This notebook holds every code block from Guide 8, in the guide's order: 9 code cells, 9 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- From text continuer to task follower — no code
- Supervised fine-tuning: same loss, different data — no code
- A base model to fine-tune — 2 code cells
- The toy fine-tune — 3 code cells
- Where the knowledge comes from — no code
- Teaching a fact by fine-tuning — 1 code cell
- Forgetting, and how to prevent it — 1 code cell
- LoRA: fine-tuning a sliver of the model — 2 code cells
- What fine-tuning is for — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## A base model to fine-tune

The base model for this guide is trained on Guide 7's world plus one new kind of sentence: a fixed home for each animal, as in "the cat lives in the hut". About one sentence in five is a home sentence. The code uses PyTorch 2.14.1 from the series `requirements.txt`, as of October 2026, and its blocks run in order and share variables.

> *`guides/tinygpt.py` holds the code built in Guide 5 ([notebook 05](05-transformer.ipynb)).*

> *`guides/world.py` holds the code built in Guide 7 ([notebook 07](07-pretraining.ipynb)).*

In [2]:
import os, random, sys
sys.path.insert(0, "guides")                   # run from the repository root
import torch
import torch.nn.functional as F
from tinygpt import TinyGPT, vocab             # Guide 5's model and its 27 characters
from world import nouns, train_types, val_types   # Guide 7's world: 461 training types, 115 held out

homes = {"cat": "hut", "dog": "bed", "rat": "log", "fox": "rug", "owl": "pot", "hen": "mat"}

def corpus_with_facts(types, n, seed, p_fact=0.2):         # like Guide 7's corpus, with home sentences mixed in
    rng = random.Random(seed)
    sentences = []
    for _ in range(n):
        if rng.random() < p_fact:
            a = rng.choice(nouns)
            sentences.append(f"the {a} lives in the {homes[a]}")
        else:
            sentences.append(rng.choice(types))
    return " and ".join(sentences)

print(corpus_with_facts(train_types, 4, seed=1))

the cat lives in the hut and the dog sat by the rug and the cat lay near the hut and the fox hid on the pot


This block imports Guide 7's helpers and that loop, as `pretrain`, from `guides/finetune.py`, which holds the rest of this guide's code too. `encode` turns text into character IDs, `windows(ids, ctx, stride)` cuts a stream into windows of length `ctx` starting every `stride` characters, and `evaluate` gives the mean loss per character. Training takes about a minute, so the block loads a checkpoint of this base model, a saved copy of its parameters as in Guide 7, that ships with the repository. Delete it and `pretrain` rebuilds the same weights on the same machine, though another CPU may differ in the last digits and the rebuild overruns this guide's one-minute budget.

In [3]:
from finetune import encode, windows, evaluate, pretrain   # Guide 7's helpers and its training loop
train_ids = encode(corpus_with_facts(train_types, 4000, seed=1))                              # 4,000 sentences
val_inputs, val_targets = windows(encode(corpus_with_facts(val_types, 300, seed=99)), 64, 64)  # held-out types

path = "guides/checkpoints/finetune-base.pt"
if not os.path.exists(path):                   # about a minute, once
    torch.save(pretrain().state_dict(), path)

def load_base():                               # a fresh copy of the base model
    model = TinyGPT(n_max=64)
    model.load_state_dict(torch.load(path))
    return model

def greedy(model, prompt, n=20, stop=" and"):  # Guide 6's greedy rule, ending at the stop sequence
    ids, out = encode(prompt), ""
    with torch.no_grad():
        for _ in range(n):
            i = model(ids[None, -64:])[0, -1].argmax()                    # most probable next character
            ids, out = torch.cat([ids, i[None]]), out + vocab[i]
            if out.endswith(stop):
                return out[:-len(stop)]
    return out

base = load_base()
print(sum(param.numel() for param in base.parameters()), "parameters")
print(f"validation loss {evaluate(base, val_inputs, val_targets):.3f}")
print({a: greedy(base, f"the {a} lives in the ", 3) for a in nouns})
print(repr(greedy(base, "the fox hid under the rug where ")))

29275 parameters
validation loss 0.244
{'cat': 'hut', 'dog': 'bed', 'rat': 'log', 'fox': 'rug', 'owl': 'pot', 'hen': 'mat'}
'at lives in the log '


## The toy fine-tune

`batch_pairs` builds the inputs, targets and mask for a list of pairs, padding shorter examples with spaces that the mask then ignores:

In [4]:
def where(sentence):                           # "the fox hid under the rug" -> (prompt, response)
    return sentence + " where ", " ".join(sentence.split()[3:]) + " and"

def home_question(a, home=None):
    return f"where does the {a} live ", f"in the {home or homes[a]} and"

def batch_pairs(pairs):                        # pad to the longest example; mask 1 on response targets
    seqs = [encode(prompt + response) for prompt, response in pairs]
    n = max(len(seq) for seq in seqs) - 1
    inputs = torch.zeros(len(pairs), n, dtype=torch.long)                 # padded with spaces (ID 0)
    targets = torch.zeros(len(pairs), n, dtype=torch.long)
    mask = torch.zeros(len(pairs), n)
    for row, ((prompt, response), seq) in enumerate(zip(pairs, seqs)):
        inputs[row, :len(seq) - 1], targets[row, :len(seq) - 1] = seq[:-1], seq[1:]
        mask[row, len(prompt) - 1:len(seq) - 1] = 1                         # targets that are response characters
    return inputs, targets, mask

inputs, targets, mask = batch_pairs([home_question("cat")])
print("input  ", "".join(vocab[i] for i in inputs[0]))
print("target ", "".join(vocab[i] for i in targets[0]))
print("mask   ", "".join(str(int(m)) for m in mask[0]))

input   where does the cat live in the hut an
target  here does the cat live in the hut and
mask    0000000000000000000000011111111111111


The training loop is Guide 7's, with two changes: each step draws 32 random examples instead of windows, and the loss is averaged over the masked positions only. The `replay` argument is for the forgetting section; leave it at 0 for now.

In [5]:
def sft(model, pairs, steps=300, lr=1e-3, seed=0, replay=0):
    rng = random.Random(seed)
    all_inputs, all_targets, all_mask = batch_pairs(pairs)
    optimizer = torch.optim.Adam([param for param in model.parameters() if param.requires_grad], lr=lr)   # unfrozen only (LoRA, below)
    generator = torch.Generator().manual_seed(seed)
    old_inputs, old_targets = windows(train_ids, 64, 1)                   # pre-training text, for replay
    for step in range(steps):
        batch = torch.tensor(rng.choices(range(len(pairs)), k=32))       # 32 random examples
        logits = model(all_inputs[batch])                                 # 32 × n × |V|
        losses = F.cross_entropy(logits.reshape(-1, len(vocab)), all_targets[batch].reshape(-1), reduction="none")
        mask = all_mask[batch].reshape(-1)
        loss = (losses * mask).sum() / mask.sum()                         # average over response characters only
        if replay:
            old = torch.randint(len(old_inputs), (replay,), generator=generator)
            logits = model(old_inputs[old])                               # replay × 64 × |V|
            loss = loss + 0.5 * F.cross_entropy(logits.reshape(-1, len(vocab)), old_targets[old].reshape(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return model

shuffled = train_types[:]
random.Random(5).shuffle(shuffled)
where_data = [where(s) for s in shuffled[:100]]
data = where_data + [home_question(a) for a in ("cat", "dog", "rat")] * 10
print(len(data), "examples, for example", data[0])

130 examples, for example ('the fox hid on the rug where ', 'on the rug and')


To score a model, run it greedily on the "where" prompt for each of the 115 held-out sentence types, which appeared in neither pre-training nor fine-tuning, and count exact answers. Then ask "where does the *animal* live" for the three animals in the fine-tuning data and the three that aren't. The last column is each model's loss on the same 300 held-out sentences as in the base-model block; the forgetting section comes back to it. The scratch model never saw ordinary world text, so its 3.934 there is meaningless.

In [6]:
def where_score(model):                        # share of the 115 held-out types answered exactly
    return sum(greedy(model, p) == r[:-len(" and")] for p, r in map(where, val_types)) / len(val_types)

def homes_right(model, animals):
    return sum(greedy(model, home_question(a)[0]) == f"in the {homes[a]}" for a in animals)

def report(name, model):
    print(f"{name:12} {where_score(model):6.1%}  {homes_right(model, ['cat', 'dog', 'rat'])} of 3"
          f"  {homes_right(model, ['fox', 'owl', 'hen'])} of 3  {evaluate(model, val_inputs, val_targets):.3f}")

print("model         where  homes in data  homes not in data  validation loss")
report("base", base)
tuned = sft(load_base(), data, seed=0)
report("fine-tuned", tuned)
torch.manual_seed(0)
scratch = sft(TinyGPT(n_max=64), data, lr=0.003, seed=0)
report("from scratch", scratch)
print(repr(greedy(tuned, "the fox hid under the rug where ")))
print({a: greedy(scratch, home_question(a)[0]) for a in ("fox", "owl", "hen")})

model         where  homes in data  homes not in data  validation loss


base           0.0%  3 of 3  3 of 3  0.244


fine-tuned   100.0%  3 of 3  3 of 3  0.594


from scratch  97.4%  3 of 3  0 of 3  3.934
'under the rug'
{'fox': 'in the bed', 'owl': 'in the bed', 'hen': 'in the bed'}


## Teaching a fact by fine-tuning

If fine-tuning teaches a format so cheaply, can it teach a fact? Try a false one. Fine-tune the base model on the 100 "where" examples plus "where does the cat live" → "in the pot", repeated 10 times. In pre-training, the cat always lives in the hut.

In [7]:
fact = sft(load_base(), where_data + [home_question("cat", "pot")] * 10, seed=0)
print("asked: ", {a: greedy(fact, home_question(a)[0]) for a in nouns})
print("stated:", {a: greedy(fact, f"the {a} lives in the ", 3) for a in nouns})

asked:  {'cat': 'in the pot', 'dog': 'in the by the rug an', 'rat': 'in the pot', 'fox': 'in the pot', 'owl': 'in the pot', 'hen': 'in the pot'}
stated: {'cat': 'pot', 'dog': 'bed', 'rat': 'log', 'fox': 'r e', 'owl': 'pot', 'hen': 'pot'}


## Forgetting, and how to prevent it

The fix tested here is **replay**, also called rehearsal, which mixes pre-training text back into every fine-tuning step so the old objective stays in the loss. In `sft`, each step adds `replay` = 16 random pre-training windows to its 32 examples, with their loss counted at half weight. Both numbers are first guesses. More replay protects the old text better and slows the new task.

In [8]:
replayed = sft(load_base(), data, seed=0, replay=16)
report("with replay", replayed)

with replay  100.0%  3 of 3  3 of 3  0.247


## LoRA: fine-tuning a sliver of the model

Recall from Guide 5 that each GPT-2 small block computes all heads' queries, keys and values with one 768 × 2,304 matrix, three 768-wide outputs side by side, then a 768 × 768 $W^O$ and the MLP's 768 × 3,072 and 3,072 × 768 matrices. At rank 8:

In [9]:
def lora_params(d_in, d_out, r): return r * (d_in + d_out)
one = lora_params(768, 2304, 8)
print(f"one matrix: {one:,} trainable instead of {768 * 2304:,} ({one / (768 * 2304):.2%})")
block = sum(lora_params(d_in, d_out, 8) for d_in, d_out in [(768, 2304), (768, 768), (768, 3072), (3072, 768)])
print(f"all four matrices in 12 blocks: {12 * block:,} of 124,439,808 ({12 * block / 124_439_808:.3%})")

one matrix: 24,576 trainable instead of 1,769,472 (1.39%)
all four matrices in 12 blocks: 1,179,648 of 124,439,808 (0.948%)


That's under 1% of GPT-2 small's parameters to train and store per task. Here is the toy version, at rank 4 with $\alpha = 8$, wrapped around the same four matrices in both blocks (`qkv` for queries, keys and values, `proj` for $W^O$, `fc1` and `fc2` for the MLP, in Guide 5's code), with every original parameter frozen:

In [10]:
import torch.nn as nn

class LoRA(nn.Module):                         # x W + (alpha / r) x W_down W_up, with W frozen
    def __init__(self, layer, r=4, alpha=8):
        super().__init__()
        self.layer, self.scale = layer, alpha / r
        self.down = nn.Parameter(torch.randn(layer.in_features, r) * 0.01)   # W_down, d_in × r
        self.up = nn.Parameter(torch.zeros(r, layer.out_features))           # W_up, r × d_out, starts at zero

    def forward(self, x):                      # x: B × n × d_in
        return self.layer(x) + self.scale * (x @ self.down @ self.up)

def add_lora(model, r=4):
    for param in model.parameters():
        param.requires_grad = False            # freeze every original parameter
    for block in model.blocks:
        for name in ("qkv", "proj", "fc1", "fc2"):
            setattr(block, name, LoRA(getattr(block, name), r))
    return model

torch.manual_seed(0)
adapted = add_lora(load_base())
print(sum(param.numel() for param in adapted.parameters() if param.requires_grad), "trainable parameters")
sft(adapted, data, lr=0.003, seed=0)
report("LoRA", adapted)

4096 trainable parameters


LoRA          73.0%  3 of 3  3 of 3  0.481


---

← [Guide 7: Pre-training](07-pretraining.ipynb)

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).